# Train a plankton classifier

Train a small supervised classification head on expert-labeled Northeast U.S. Shelf Imaging FlowCytobot crops. A frozen MobileNetV3-Small encoder supplies ImageNet features. Labels from the training split determine the head and feature scaling. Whole acquisition bins stay on one side of the train–validation boundary.

On the OOI JupyterHub, choose the **SIG.AI Ocean** kernel. For another JupyterLab, keep this notebook beside the two repository scripts or set the repository path below. Running all cells reuses the saved model when it exists. To train a new candidate, change the run name. The first training run needs internet access for the initialization weights and any missing source data.

[Expert-labeled dataset and citation](https://huggingface.co/datasets/sosiklab/NES-plankton-IFCB-classifier-2022-dataset) · [TorchVision backbone and ImageNet weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v3_small.html)

In [ ]:
import csv
import json
import os
from pathlib import Path
import subprocess
import sys

ROOT = Path(os.environ.get("OCEAN_WORKSPACE_ROOT", Path.home() / "data/ocean-platform")).expanduser()
DATASET = ROOT / "datasets" / "whoi-nes-2022" / "pilot"
SOURCE = DATASET.parent / "source" / "data"
RUN_NAME = "mobilenet-ifcb-pilot"
RUN = ROOT / "models" / RUN_NAME
CHECKPOINT = RUN / "model.pt"
# Set REPO directly if the repository is elsewhere.
candidates = [Path.cwd(), Path.home() / "marinemodels", *Path.cwd().parents]
REPO = next((p for p in candidates if (p / "plankton_workspace.py").is_file()
             and (p / "prepare_plankton_training_data.py").is_file()), None)
if REPO is None:
    REPO = next((p for p in candidates if (p / "plankton_workspace.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Set REPO to the marinemodels directory containing both Python scripts.")
print("Dataset:", DATASET)
print("Model:", CHECKPOINT, "·", "ready to load" if CHECKPOINT.is_file() else "first training run")

## Prepare the runtime

The existing kernel environment is used when its packages are available. Otherwise this cell installs missing research dependencies into a separate environment under the data folder. The PyTorch device check reports the actual accelerator selected. CPU training is also supported.

In [ ]:
import venv

probe = "import torch, torchvision, PIL, pyarrow"
python = sys.executable
if subprocess.run([python, "-c", probe], capture_output=True).returncode:
    env_dir = ROOT / "envs" / ("plankton-training-py" + str(sys.version_info.major) + str(sys.version_info.minor))
    python = str(env_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python"))
    if not Path(python).exists():
        venv.EnvBuilder(with_pip=True, system_site_packages=True).create(env_dir)
    if subprocess.run([python, "-c", probe], capture_output=True).returncode:
        subprocess.run([python, "-m", "pip", "install", "torch", "torchvision", "Pillow", "pyarrow"], check=True)
    subprocess.run([python, "-c", probe], check=True)
subprocess.run([python, "-c", "import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"], check=True)

## Train or load the released run

The full pilot selects 128 training crops and 25 validation crops per class across ten classes from the published shards. The training script rejects overlapping bins, repeated image identifiers, cross-split image hashes and altered PNG files. It trains a class-weighted linear head for 40 fixed epochs. Validation does not determine the stopping epoch or the feature scaling.

A completed run contains the model and its preprocessing, validation metrics, and one prediction per held-out image. An existing run directory is preserved.

In [ ]:
if not CHECKPOINT.is_file():
    if not (DATASET / "manifest.csv").is_file():
        if not (REPO / "prepare_plankton_training_data.py").is_file():
            raise FileNotFoundError("Set REPO to a checkout that contains prepare_plankton_training_data.py.")
        prepare = [python, str(REPO / "prepare_plankton_training_data.py"), "--output", str(DATASET), "--all-shards"]
        if SOURCE.is_dir():
            prepare.extend(["--source-dir", str(SOURCE)])
        subprocess.run(prepare, check=True)
    subprocess.run([python, str(REPO / "plankton_workspace.py"), "train",
                    "--dataset", str(DATASET), "--output", str(RUN),
                    "--epochs", "40", "--batch-size", "24", "--device", "auto"], check=True)
else:
    print("Using the existing checkpoint:", CHECKPOINT)

## Inspect the held-out result

Macro-F1 gives each supported class equal weight. Balanced accuracy averages per-class recall. Compare these measures to the training-majority baseline and inspect the class support before interpreting a difference. This pilot samples selected classes from the WHOI reference collection. The validation figures apply to that selection. Assessing Pioneer imagery requires an independent expert audit.

In [ ]:
metrics = json.loads((RUN / "metrics.json").read_text())
for key in ["train_images", "train_groups", "validation_images", "validation_groups",
            "accuracy", "balanced_accuracy", "macro_f1", "majority_baseline", "accelerator", "manifest_sha256"]:
    print(f"{key}: {metrics[key]}")
print()
print(f"{'Label':36} {'Support':>7} {'Precision':>10} {'Recall':>8} {'F1':>8}")
for row in metrics["per_class"]:
    def score(name):
        return "—" if row[name] is None else f"{row[name]:.3f}"
    print(f"{row['label']:36} {row['support']:7} {score('precision'):>10} {score('recall'):>8} {score('f1'):>8}")
print("Per-image results:", RUN / "validation_predictions.csv")

## Classify a crop with the saved checkpoint

Set **IMAGE** to your own microscope PNG to examine its top suggestions. The default is an original Pioneer crop from the public replay. This command loads the saved model without modifying its weights or downloading ImageNet again. Softmax values express preferences within the pilot vocabulary. Out-of-vocabulary organisms and debris can still receive a high score.

In [ ]:
IMAGE = REPO / "site" / "assets" / "D20260201T002546_IFCB200_00139.png"
if not IMAGE.is_file() and (DATASET / "manifest.csv").is_file():
    with (DATASET / "manifest.csv").open() as handle:
        IMAGE = DATASET / next(r for r in csv.DictReader(handle) if r["split"] == "validation")["path"]
if not IMAGE.is_file():
    raise FileNotFoundError("Set IMAGE to a local microscope crop.")
command = [python, str(REPO / "plankton_workspace.py"), "predict", "--checkpoint", str(CHECKPOINT), "--image", str(IMAGE)]
result = json.loads(subprocess.check_output(command, text=True))
print(IMAGE.name, result["status"])
for item in result["predictions"]:
    print(f"{item['label']:36} {item['softmax']:.4f}")
print("Image SHA-256:", result["input_sha256"])